# grads-dict-accumulate-parents — worked example 2: grads dict: shared parent in y = x + x accumulates both contributions

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `grads-dict-accumulate-parents`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

A classic example of the accumulation requirement: `y = x + x` means `x` is a parent of `y` at BOTH argument positions (argnum 0 and argnum 1). Each position contributes an independent backward path with its own gradient. The final `dL/dx` is the SUM of both. Without accumulation — if we overwrote instead — we'd discard one contribution and compute a gradient that's half the correct value.

## Worked solution

**Step 1 — set up the problem.** We have a scalar `x` and compute `y = x + x`. The correct gradient `dL/dx` at loss `L = y^2 / 2` with chain rule is `dL/dy * 2 = y * 2 = (2x) * 2 = 4x`.

**Step 2 — simulate the two-path backward.** Argnum 0 contributes `grad_y * 1` (derivative of `x+x` wrt first argument). Argnum 1 also contributes `grad_y * 1`. Both paths reach `x`.

**Step 3 — accumulate both.** Call `accumulate_into_grads` twice with `x` as the parent each time. The second call adds to the first.

**Step 4 — compare against PyTorch.** Create the same computation with real autograd and verify `x.grad` matches our accumulated value.

In [ ]:
import torch as t

t.manual_seed(0)

class Node:
    def __init__(self, name):
        self.name = name

def accumulate_into_grads(grads, contributions):
    for parent, g in contributions:
        grads[parent] = grads.get(parent, 0) + g

# Scalar x; y = x + x; loss = y^2 / 2
x_val = t.tensor(3.0)
y_val = x_val + x_val      # y = 6
loss_val = y_val ** 2 / 2  # loss = 18

# grad_y = dL/dy = y = 6
grad_y = y_val.clone()

# Simulate add backward: both argnums contribute grad_y * 1
x_node = Node('x')
grads = {}
accumulate_into_grads(grads, [(x_node, grad_y * 1)])  # argnum 0
accumulate_into_grads(grads, [(x_node, grad_y * 1)])  # argnum 1

print(f"Our accumulated dL/dx: {grads[x_node].item()}")  # expect 12.0

# Verify with real autograd
x_real = t.tensor(3.0, requires_grad=True)
y_real = x_real + x_real
loss_real = y_real ** 2 / 2
loss_real.backward()
print(f"PyTorch autograd dL/dx: {x_real.grad.item()}")  # expect 12.0

assert abs(grads[x_node].item() - x_real.grad.item()) < 1e-5
print("Matches! Both paths were correctly summed.")